# 32 — Umpire error profiles

What each umpire struggles with. All the logic lives in `src/umpire_profiles.py`.

Each season is graded against its own strike zone (old zone 2021–2025, new ABS zone 2026), and an umpire is only compared with the umpires of the same season. Nothing is standardised.

**Words used here**

- **False strike**: a true ball called a strike (helps the pitcher). **False ball**: a true strike called a ball (helps the hitter).
- **Net strikes** = false strikes − false balls. Positive means more strikes called than the zone has.
- **Lean** = an umpire's net strikes minus the league's, per game. Positive = he calls more strikes than the league, negative = more balls. It is *systematic* when its 95% interval does not include zero.
- **Zone shape** = the same sum inside one region: where his zone is bigger or tighter than the league's.
- **The line** = where the right call flips from strike to ball (the edge of the ball just touching the zone).

In [ ]:
import sys

import pandas as pd
from IPython.display import display

sys.path.insert(0, "../src")
import plot_style
import umpire_profiles
import umpire_tiers

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

tables = umpire_profiles.build()
umpire_tiers.save(tables)

profiles = tables["umpire_profiles"]
splits = tables["umpire_split_accuracy"]
league = tables["league_split_accuracy"]
stability = tables["profile_stability"]
showcase = tables["showcase_umpires"]
SEASON = int(profiles["season"].max())

## 1. The league: two kinds of miss, season by season

All umpires together. `net_strikes_per_1000` above zero means umpires called more strikes than the zone has.

In [ ]:
overall = league[league["split"] == "all calls"]
display(
    overall[
        [
            "season",
            "zone",
            "called",
            "accuracy",
            "false_strikes",
            "false_strike_rate",
            "false_balls",
            "false_ball_rate",
            "net_strikes_per_1000",
        ]
    ].round(4)
)

## 2. The count changes the call

Same pitch, different count. `near_line_strike_rate` is the share called a strike among pitches within 1 inch of the line, where about half are true strikes (`near_line_true_share`).

Two strikes = 0-2, 1-2, 2-2. Three balls = 3-0, 3-1. Full count = 3-2.

In [ ]:
situations = league[
    (league["split"] == "count_situation") & (league["level"] != "unknown")
]
columns = [
    "level",
    "called",
    "accuracy",
    "false_strike_rate",
    "false_ball_rate",
    "near_line",
    "near_line_strike_rate",
    "near_line_true_share",
]
display(situations.loc[situations["season"] == SEASON, columns].round(4))

fig = umpire_profiles.plot_borderline_by_count(league)
path = plot_style.save_figure(fig, "borderline_calls_by_count")

All twelve counts, latest season.

In [ ]:
counts = league[
    (league["split"] == "count")
    & (league["season"] == SEASON)
    & (league["level"] != "unknown")
]
display(
    counts[
        [
            "level",
            "called",
            "share_of_called",
            "accuracy",
            "false_strike_rate",
            "false_ball_rate",
            "near_line_strike_rate",
        ]
    ].round(4)
)

### Two ways to say the two-strike number

- **Per true ball** (`false_strike_rate`): of the true balls, how many were called strikes.
- **Per called strike** (`wrong_strike_share`): of the pitches called a strike, how many were really balls.

With two strikes the first goes *down* and the second goes *up*. Both are right. Hitters swing at most strikes when they have two strikes, so the pitches left for the umpire are mostly balls: few of them get called strikes, but those few are a bigger part of the small number of called third strikes.

In [ ]:
by_count = league[(league["split"] == "count") & (league["level"] != "unknown")]
by_count = by_count.assign(two_strikes=by_count["level"].str.endswith("-2"))
sums = by_count.groupby(["season", "two_strikes"])[
    ["called_strikes", "false_strikes", "true_balls"]
].sum()
sums["false_strike_rate"] = sums["false_strikes"] / sums["true_balls"]
sums["wrong_strike_share"] = sums["false_strikes"] / sums["called_strikes"]
display(
    sums[["false_strike_rate", "wrong_strike_share"]]
    .unstack("two_strikes")
    .round(4)
)

## 3. Pitch type, batter side and distance from the line

League numbers, the last old-zone season beside the new-zone season. `borderline` = the centre of the ball within 2 inches of the line; `clear` = everything else.

In [ ]:
recent = league[league["season"] >= SEASON - 1]
for split in ("pitch_group", "batter", "line"):
    rows = recent[recent["split"] == split]
    display(
        rows[
            [
                "split",
                "level",
                "season",
                "called",
                "share_of_called",
                "accuracy",
                "false_strike_rate",
                "false_ball_rate",
                "net_strikes_per_1000",
            ]
        ].round(4)
    )

distance = league[league["split"] == "distance"]
print("Accuracy by distance from the line, every season")
display(
    distance.pivot(index=["order", "level"], columns="season", values="accuracy")
    .round(4)
    .droplevel("order")
)

## 4. Each umpire's lean

`lean_per_game` is how many more strike calls a game he makes than a league-average umpire would on the same number of pitches (negative = more ball calls). The low and high columns are the 95% interval.

In [ ]:
pool = profiles[(profiles["season"] == SEASON) & profiles["qualifies"]]
print("Umpires by lean,", SEASON)
display(pool["lean_side"].value_counts().to_frame("umpires"))

show = [
    "official_name",
    "tier",
    "plate_games",
    "accuracy",
    "false_strike_rate",
    "false_ball_rate",
    "lean_per_game",
    "lean_low_per_game",
    "lean_high_per_game",
    "lean_side",
]
print("Most strike-leaning,", SEASON)
display(pool.sort_values("lean_low_per_1000", ascending=False)[show].head(10).round(3))
print("Most ball-leaning,", SEASON)
display(pool.sort_values("lean_high_per_1000")[show].head(10).round(3))

print("Lean by tier,", SEASON)
display(pd.crosstab(pool["tier"], pool["lean_side"]))

## 5. Which traits are real?

A real trait shows up again the next season; noise does not. Each row is the correlation between a trait in one season and the same trait a year later, across umpires who qualify in both.

- `aal_pp` (overall accuracy, what the tiers are built on) is the yardstick.
- `beyond_accuracy_correlation` asks the same thing after taking out what overall accuracy already explains. If it drops to about zero, that trait was just overall accuracy seen from another angle.

In [ ]:
columns = [
    "trait",
    "all_pairs",
    "all_correlation",
    "old_zone_correlation",
    "old_to_new_correlation",
    "beyond_accuracy_correlation",
]
for group, rows in stability.groupby("group", sort=False):
    print(group)
    display(rows[columns].round(3))

## 6. Zone shape: where each umpire's zone is bigger or tighter

`zone_..._per_game` = extra strike calls per game in that region compared with the league (negative = extra ball calls). `widest_region` and `tightest_region` are the regions that stand out most; `..._secure` means it still stands out after allowing for the nine regions looked through.

In [ ]:
zone_columns = [name for name in profiles.columns if name.startswith("zone_")]
flags = pool[["widest_region_secure", "tightest_region_secure"]]
print(
    f"{SEASON}: {int(flags.any(axis=1).sum())} of {len(pool)} umpires have a region "
    f"clearly bigger or tighter than the league's "
    f"({int(flags['widest_region_secure'].sum())} wide, "
    f"{int(flags['tightest_region_secure'].sum())} tight)"
)
print("Where the secure wide and tight regions are")
display(
    pd.DataFrame(
        {
            "wide": pool.loc[pool["widest_region_secure"], "widest_region"].value_counts(),
            "tight": pool.loc[pool["tightest_region_secure"], "tightest_region"].value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
)

## 7. Four example umpires

Best and worst are the first and last umpire in the tier ranking. The most strike-leaning and most ball-leaning umpires are picked on the cautious end of their interval, so a short season cannot win on luck. These are the four umpires for the heatmaps (J11).

In [ ]:
display(
    showcase[
        [
            "role",
            "official_id",
            "official_name",
            "tier",
            "plate_games",
            "accuracy",
            "false_strike_rate",
            "false_ball_rate",
            "lean_per_game",
            "lean_low_per_game",
            "lean_high_per_game",
        ]
    ].round(3)
)
for pick in showcase.itertuples():
    print(f"{pick.role}: {pick.official_name}. {pick.profile_note}")

print("\nZone shape of the four: extra strike calls per game, by region")
four = profiles[
    (profiles["season"] == SEASON)
    & profiles["official_id"].isin(showcase["official_id"])
]
display(
    four.set_index("official_name")
    .loc[showcase["official_name"], zone_columns]
    .rename(columns=lambda name: name[5:-9])
    .T.round(2)
)

fig = umpire_profiles.plot_lean_vs_accuracy(profiles, showcase)
path = plot_style.save_figure(fig, "umpire_lean_vs_accuracy")

## Takeaways

Each season is measured against its own zone.

1. **Two kinds of miss.** In 2026 umpires made 12,967 false strikes (5.3 a game, 5.0% of true balls) and 11,276 false balls (4.6 a game, 10.0% of true strikes). Since 2021 false strikes fell from 17,665 to 12,967 while false balls stayed flat (11,323 → 11,276).
2. **2025 is the odd season.** It is the only one where umpires called fewer strikes than the zone has (net −9.2 per 1,000 calls; 11,385 false strikes, 14,778 false balls). That is the year MLB shrank the buffer it uses to grade umpires from 2 inches to ¾ inch (reported by The Athletic and CBS Sports in May 2025, confirmed by an MLB official). The timing fits, but we have not tested the link.
3. **The lean is a real trait.** It carries over from one season to the next at r = 0.56 (overall accuracy: 0.62), and still at 0.53 after overall accuracy is taken out. In 2026, 25 of 89 umpires have a systematic lean: 13 to strikes, 12 to balls. The range is from 3.0 more ball calls a game (Willie Traynor) to 3.4 more strike calls a game (Ron Kulpa).
4. **Strike-leaning umpires are less accurate.** In Tiers 4–5, 8 of 22 umpires lean to strikes and none to balls. In Tiers 1–2, 5 of 22 lean to balls and 1 to strikes.
5. **Zone shape is real too.** Extra strike calls per game in a region carry over at r = 0.26 (high-middle) to 0.59 (mid-outside). In 2026, 40 of 89 umpires have at least one region clearly bigger (27) or tighter (17) than the league's.
6. **Accuracy by pitch type, count or region is mostly not a personal trait.** Once overall accuracy is taken out, the carry-over is between −0.07 and 0.25 (highest: mid-outside). So we do not say an umpire is "bad at breaking balls" or "bad with two strikes".
7. **The count effect belongs to every umpire.** A pitch within 1 inch of the line is called a strike 55% of the time on the first pitch, 56% with three balls and 38% with two strikes (2026). In 2021–2025 it was 55%, 58% and 35%, so the ABS season barely changed it.
8. **Two ways to say the two-strike number.** Per true ball, umpires call fewer false strikes with two strikes (2.0% against 6.3% in other counts, 2026). Per called strike, more of them are wrong (15.7% against 10.9% in 2026; 20.8% against 14.6% in 2021).
9. **Coin-flip pitches did not get easier.** Accuracy on pitches within 2 inches of the line was 73.0% in 2025 and 72.9% in 2026. On all other pitches it went from 98.5% to 99.0%.
10. **Four example umpires for the heatmaps:** Jansen Visconti (best), Bruce Dreckman (worst), Rob Drake (most strike-leaning, +3.2 strike calls a game) and Willie Traynor (most ball-leaning, 3.0 more ball calls a game).